# Urban Scene Understanding

ResNet-based semantic and instance segmentation on Cityscapes. Compare FCN and DeepLabV3, evaluate horizontal-flip augmentation, and analyse Mask R-CNN errors.

This notebook contains data preparation, training, evaluation, and exploratory analysis. Historical results are documented in `docs/experiments.md`; checkpoints and data must be supplied separately.


## 0. Setup
Use a Colab GPU runtime. The install cell keeps Colab's PyTorch and TorchVision pair and adds the Cityscapes evaluator.

Accept the [Cityscapes terms](https://www.cityscapes-dataset.com/register/). Download `leftImg8bit_trainvaltest.zip` and `gtFine_trainvaltest.zip`, then place both archives in one Drive folder. The next section unpacks them onto the runtime disk.

Supply your own completed run directories under `OUTPUT_ROOT` to reload results. No checkpoints are bundled.

Each run stores package versions, hardware details, its configuration and source commit. Reproduction uses its `pip-freeze.txt` with a compatible Python and CUDA environment.


In [ ]:
# Avoid the obsolete typing backport declared by cityscapesScripts.
%pip install -q "numpy>=1.26,<2.3" pandas scipy matplotlib Pillow appdirs pyquaternion coloredlogs tqdm requests
%pip install -q --no-deps cityscapesScripts==2.2.4


In [ ]:
import copy
import hashlib
import importlib.metadata
import json
import platform
import os
import random
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import functional as TF
from torchvision.models import ResNet50_Weights
from torchvision.models.segmentation import fcn_resnet50, deeplabv3_resnet50
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from cityscapesscripts.helpers.labels import labels as city_labels

# Report figures follow the reference paper's physical sizes and typography.
REPORT_SINGLE_SIZE = (3.45, 2.55)
REPORT_WIDE_SIZE = (7.4, 1.75)
REPORT_DOUBLE_SIZE = (7.4, 2.55)
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8.5,
    'axes.titlesize': 9.5,
    'axes.labelsize': 8.5,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 7.5,
    'legend.fontsize': 8.5,
    'axes.linewidth': 0.8,
    'savefig.dpi': 200,
})

def style_report_axis(axis, grid=False):
    axis.tick_params(axis='both', labelsize=7.5, length=3)
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)
    if grid:
        axis.grid(axis='y', color='#d0d0d0', linewidth=0.7, zorder=0)
    return axis

CONFIG = {
    "seed": 9444,
    "height": 256, "width": 512,
    "epochs": 5, "smoke_epochs": 1, "batch_size": 2, "workers": 2,
    "learning_rate": 1e-4, "weight_decay": 1e-4,
    "pretrained": True,
    "smoke": False, "smoke_train": 8, "smoke_val": 4,
    # Profile mode times 1 epoch on the full split. Exclude its scores from the report.
    "profile": False,
    "instance_score_threshold": 0.05, "mask_threshold": 0.5,
    "detections_per_image": 50,
}
# Final instance analysis uses the highest tested cap so predictions are not truncated.
REPORT_DETECTION_CAP = 200
# 407a0ff produced the reported runs. Use the current HEAD for new training.
PROVENANCE = {"source_commit": "407a0ff"}
# False rebuilds results from finished runs; True starts training.
RUN_TRAINING = False
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
DATA_ROOT = Path(os.environ.get('CITYSCAPES_ROOT', '/content/cityscapes'))
OUTPUT_ROOT = Path(os.environ.get('URBAN_SCENE_OUTPUT_ROOT', '/content/drive/MyDrive/urban-scene-runs' if MOUNT_DRIVE else 'outputs/urban-scene-runs'))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# Store temporary evaluator masks on local disk, not Drive.
SCRATCH_ROOT = Path('/content/instance-eval') if MOUNT_DRIVE else Path('outputs/instance-eval')
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_all(CONFIG['seed'])
ENVIRONMENT = {
    'python': platform.python_version(), 'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'cityscapesScripts': importlib.metadata.version('cityscapesScripts'),
    'device': str(DEVICE), 'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
}
print(json.dumps(ENVIRONMENT, indent=2))
# Smoke mode takes precedence over profile mode.
MODE = 'smoke' if CONFIG['smoke'] else ('profile' if CONFIG['profile'] else 'full')
print('Mode:', MODE, '| Training enabled:', RUN_TRAINING)
if MODE == 'profile':
    print(f"Profiling 1 epoch on the full split to size the budget; "
          f"set profile=False for a run of {CONFIG['epochs']} epochs.")


### Copy the dataset to the runtime disk
Set `ARCHIVE_DIR` to the Drive folder containing both archives. This cell unpacks them to `DATA_ROOT` and skips files already present. Allow about 11 GB of runtime storage. Reading the images from the runtime disk is faster than reading them from Drive during training.


In [ ]:
# Drive folder holding leftImg8bit_trainvaltest.zip and gtFine_trainvaltest.zip.
ARCHIVE_DIR = Path(os.environ.get('CITYSCAPES_ARCHIVE_DIR', '/content/drive/MyDrive/cityscapes-archives'))
ARCHIVES = ['leftImg8bit_trainvaltest.zip', 'gtFine_trainvaltest.zip']

def dataset_ready(root):
    # The official archives unpack to leftImg8bit/<split>/<city> and gtFine/<split>/<city>.
    return all(any((root / tree / split).glob('*/*.png'))
               for tree in ('leftImg8bit', 'gtFine') for split in ('train', 'val'))

if dataset_ready(DATA_ROOT):
    print('Dataset already staged at', DATA_ROOT)
else:
    present = [name for name in ARCHIVES if (ARCHIVE_DIR / name).exists()]
    missing = [name for name in ARCHIVES if name not in present]
    if missing:
        raise FileNotFoundError(
            f'Missing {missing} under {ARCHIVE_DIR} (found {present or "nothing"}). '
            'Both archives are needed: leftImg8bit holds the images and gtFine the '
            'annotations. Download them from https://www.cityscapes-dataset.com/downloads/ '
            'and upload them there, or point ARCHIVE_DIR at the folder that holds them.')
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    for name in ARCHIVES:
        print('Extracting', name, flush=True)
        # unzip is faster than zipfile here and -n keeps a partial extraction resumable.
        exit_code = os.system(f'unzip -q -n "{ARCHIVE_DIR / name}" -d "{DATA_ROOT}"')
        if exit_code != 0:
            raise RuntimeError(f'Extraction of {name} failed with exit code {exit_code}.')
    if not dataset_ready(DATA_ROOT):
        raise RuntimeError(f'{DATA_ROOT} is still incomplete after extraction; check the archives.')
    print('Dataset staged at', DATA_ROOT)


## 1. Problem and experiment design
Semantic segmentation assigns a class to each pixel. Instance segmentation also separates objects within the same class. Semantic maps describe road layout, while instance masks retain separate cars, riders and pedestrians. We run the tasks separately and compare errors for each class.

1. **RQ1.** How does DeepLabV3 compare with FCN at the same input size and training budget?
2. **RQ2.** How does horizontal flipping affect DeepLabV3?
3. **RQ3.** Which classes are difficult for Mask R-CNN, and do the semantic models show similar errors for those classes?

| ID | Task | Model | Flip | Role |
|---|---|---|---|---|
| S0 | Semantic | FCN-ResNet50 | Yes | Baseline |
| S1 | Semantic | DeepLabV3-ResNet50 | Yes | Architecture comparison |
| S2 | Semantic | DeepLabV3-ResNet50 | No | Flip ablation |
| I0 | Instance | Mask R-CNN-ResNet50-FPN | Yes | Instance baseline |

TorchVision supplies the models and pretrained weights. The semantic models share ImageNet backbone weights; Mask R-CNN starts from COCO weights with new prediction heads. Our work covers the Cityscapes data pipeline, training, official evaluation and error analysis. Semantic mIoU and instance AP measure different tasks.


## 2. Data and labels
We use Cityscapes fine annotations: 2,975 training images and 500 validation images. The test split has 1,525 images with hidden labels. Training uses the training split; checkpoint selection and all reported scores use validation.

Semantic labels use the 19 evaluation classes with `trainId` values from 0 to 18. Other pixels have value 255 and are ignored in the loss and metric. Instance labels use eight thing classes and background index 0. An instance ID is `labelId * 1000 + objectId`. Values below 1000 include group regions and are excluded as individual targets. The official evaluator handles crowd and void regions. Mask R-CNN treats the excluded group regions as background during training.

RGB images use bilinear resizing, while label maps use nearest neighbour resizing. The same geometry is applied to RGB images and both label maps. Bounding boxes are rebuilt after transforms. Before evaluation, predictions return to the original image size.


In [ ]:
SEMANTIC_LABELS = sorted([x for x in city_labels if 0 <= x.trainId < 19], key=lambda x: x.trainId)
THING_LABELS = [x for x in city_labels if x.hasInstances and not x.ignoreInEval]
CLASS_NAMES = [x.name for x in SEMANTIC_LABELS]
LUT = np.full(256, 255, dtype=np.uint8)
for label in SEMANTIC_LABELS:
    LUT[label.id] = label.trainId
THING_TO_MODEL = {x.id: i + 1 for i, x in enumerate(THING_LABELS)}
MODEL_TO_THING = {v: k for k, v in THING_TO_MODEL.items()}
PALETTE = np.array([x.color for x in SEMANTIC_LABELS] + [(0, 0, 0)], dtype=np.uint8)

def make_records(split):
    records = []
    for image in sorted((DATA_ROOT / 'leftImg8bit' / split).glob('*/*_leftImg8bit.png')):
        key = image.name.removesuffix('_leftImg8bit.png')
        base = DATA_ROOT / 'gtFine' / split / image.parent.name
        semantic = base / f'{key}_gtFine_labelIds.png'
        instance = base / f'{key}_gtFine_instanceIds.png'
        if not semantic.exists() or not instance.exists():
            raise FileNotFoundError(f'Missing fine annotations for {image}')
        records.append({'key': key, 'city': image.parent.name, 'image': str(image),
                        'semantic': str(semantic), 'instance': str(instance)})
    if not records:
        raise FileNotFoundError(f'No {split} images found under {DATA_ROOT}; extract the official archives first.')
    return records

all_train, all_val = make_records('train'), make_records('val')
assert not ({r['key'] for r in all_train} & {r['key'] for r in all_val})
assert not ({r['city'] for r in all_train} & {r['city'] for r in all_val})
if not CONFIG['smoke']:
    assert (len(all_train), len(all_val)) == (2975, 500), 'Full runs require complete official splits.'
rng = random.Random(CONFIG['seed'])
train_records = rng.sample(all_train, min(CONFIG['smoke_train'], len(all_train))) if CONFIG['smoke'] else all_train
val_records = rng.sample(all_val, min(CONFIG['smoke_val'], len(all_val))) if CONFIG['smoke'] else all_val
SPLIT_MANIFEST = {'train': train_records, 'val': val_records}
SPLIT_HASH = hashlib.sha256(json.dumps(SPLIT_MANIFEST, sort_keys=True).encode()).hexdigest()
print('Training images:', len(train_records), '| Validation images:', len(val_records))
print('Split fingerprint:', SPLIT_HASH)

class UrbanDataset(Dataset):
    def __init__(self, records, task, augment=False):
        self.records, self.task, self.augment = records, task, augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        image = Image.open(row['image']).convert('RGB')
        annotation = Image.open(row['semantic' if self.task == 'semantic' else 'instance'])
        size = (CONFIG['width'], CONFIG['height'])
        image = image.resize(size, Image.Resampling.BILINEAR)
        annotation = annotation.resize(size, Image.Resampling.NEAREST)
        if self.augment and random.random() < 0.5:
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            annotation = annotation.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        image = TF.to_tensor(image)
        raw = np.array(annotation, dtype=np.int64)
        if self.task == 'semantic':
            target = torch.from_numpy(LUT[raw].astype(np.int64))
            image = TF.normalize(image, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            return image, target
        masks, boxes, classes = [], [], []
        for instance_id in np.unique(raw):
            label_id = int(instance_id) // 1000
            if instance_id < 1000 or label_id not in THING_TO_MODEL:
                continue
            mask = raw == instance_id
            ys, xs = np.where(mask)
            masks.append(mask)
            boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
            classes.append(THING_TO_MODEL[label_id])
        h, w = raw.shape
        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            'labels': torch.tensor(classes, dtype=torch.int64),
            'masks': torch.from_numpy(np.stack(masks).astype(np.uint8)) if masks else torch.zeros((0, h, w), dtype=torch.uint8),
            'image_id': torch.tensor(index),
            'area': torch.tensor([int(m.sum()) for m in masks], dtype=torch.float32),
            'iscrowd': torch.zeros(len(masks), dtype=torch.int64),
        }
        return image, target

def detection_collate(batch):
    return tuple(zip(*batch))

def make_loader(task, augment):
    generator = torch.Generator().manual_seed(CONFIG['seed'])
    return DataLoader(UrbanDataset(train_records, task, augment),
                      batch_size=CONFIG['batch_size'], shuffle=True,
                      num_workers=CONFIG['workers'], generator=generator,
                      collate_fn=detection_collate if task == 'instance' else None)


## 3. Explore the data
Show the RGB image, semantic labels and instance labels. Count training pixels and instances by class. Pixel counts differ by more than two orders of magnitude, and several evaluated classes contain narrow objects. The counts below use the full split only in a full run.


In [ ]:
def semantic_rgb(target):
    return PALETTE[np.where(target == 255, 19, target)]

row = train_records[0]
raw_sem = np.array(Image.open(row['semantic']))
raw_inst = np.array(Image.open(row['instance']))
fig, axes = plt.subplots(1, 3, figsize=REPORT_WIDE_SIZE)
axes[0].imshow(Image.open(row['image'])); axes[0].set_title('Input image')
axes[1].imshow(semantic_rgb(LUT[raw_sem])); axes[1].set_title('Semantic ground truth')
instance_view = np.zeros((*raw_inst.shape, 3), dtype=np.uint8)
color_rng = np.random.default_rng(CONFIG['seed'])
for ident in np.unique(raw_inst):
    if ident >= 1000 and int(ident) // 1000 in THING_TO_MODEL:
        instance_view[raw_inst == ident] = color_rng.integers(30, 256, size=3)
axes[2].imshow(instance_view); axes[2].set_title('Individual thing instances')
for ax in axes: ax.axis('off')
plt.tight_layout(pad=0.35, w_pad=0.35); plt.show()

pixel_counts = np.zeros(19, dtype=np.int64)
instance_counts = {x.name: 0 for x in THING_LABELS}
for row in train_records:
    target = LUT[np.array(Image.open(row['semantic']))]
    pixel_counts += np.bincount(target[target != 255], minlength=19)
    ids = np.unique(np.array(Image.open(row['instance'])))
    for label in THING_LABELS:
        instance_counts[label.name] += sum(int(i) >= 1000 and int(i) // 1000 == label.id for i in ids)
eda = pd.DataFrame({'class': CLASS_NAMES, 'pixels': pixel_counts})
figure, axis = plt.subplots(figsize=REPORT_SINGLE_SIZE)
axis.bar(eda['class'], eda['pixels'], color='#1f77b4', width=0.72, zorder=2)
axis.set_yscale('log')
axis.set_title('Training pixels per class', pad=5)
axis.set_xlabel('Class'); axis.set_ylabel('Pixels (log scale)')
axis.tick_params(axis='x', rotation=90)
style_report_axis(axis, grid=True)
figure.tight_layout(pad=0.45)
# Save report figures to Drive.
FIGURE_DIR = OUTPUT_ROOT / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
if not CONFIG['smoke']:
    figure.savefig(FIGURE_DIR / 'class-distribution.png', bbox_inches='tight', pad_inches=0.035)
plt.show()
instances = pd.DataFrame(instance_counts.items(), columns=['class', 'individual_instances'])
display(instances)
if not CONFIG['smoke']:
    eda.to_csv(FIGURE_DIR / 'class-distribution.csv', index=False)
    instances.to_csv(FIGURE_DIR / 'instance-counts.csv', index=False)


### Exploration findings
Training pixel counts differ by more than two orders of magnitude. Road and building dominate the pixel count, while pole and traffic light are thin at 256x512 input. The instance labels also contain many small people, riders, bicycles and motorcycles. Pixel accuracy alone therefore misses important errors, so we also use IoU for each class, AP and error analysis.

## 4. Models and evaluation
FCN and DeepLabV3 predict 19 class scores per pixel. Both use cross entropy with `ignore_index=255`. Their BatchNorm statistics stay fixed because batches are small and the last batch contains one image. Mask R-CNN trains its classifier, box, proposal and mask heads for eight thing classes plus background.

FCN is the faster semantic baseline. DeepLabV3 uses atrous spatial pyramid pooling to test multiscale context under the same training budget and ResNet50 backbone. Mask R-CNN is the instance baseline because it predicts boxes, classes and masks and can be scored with the official Cityscapes evaluator.

We select semantic checkpoints by mIoU from a 19x19 confusion matrix. Void pixels are ignored, and classes with zero union are left out of the mean. We also record IoU for each class and pixel accuracy. We select the instance checkpoint by official Cityscapes AP and record AP50. Both tasks are scored against validation labels at their original resolution. Instance masks are resized before thresholding.

Each run saves its split, environment, history, checkpoints and evaluator output. `last_validation` holds the latest epoch output; `best_validation` holds the selected checkpoint output. Checkpoints include optimiser and random states. Resume uses only a run with the same configuration and split.

Instance scoring writes detection masks at full resolution to the runtime disk and removes them afterwards. Scores and evaluator exports are saved to Drive. Ground truth statistics are cached by split.


In [ ]:
EXPERIMENTS = {
    'S0': {'task': 'semantic', 'model': 'fcn', 'augment': True},
    'S1': {'task': 'semantic', 'model': 'deeplab', 'augment': True},
    'S2': {'task': 'semantic', 'model': 'deeplab', 'augment': False},
    'I0': {'task': 'instance', 'model': 'maskrcnn', 'augment': True},
}

def build_model(spec):
    # Match checkpoint initialisation because it changes Mask R-CNN normalisation layers.
    pretrained = CONFIG['pretrained']
    if spec['task'] == 'semantic':
        constructor = fcn_resnet50 if spec['model'] == 'fcn' else deeplabv3_resnet50
        return constructor(weights=None,
                           weights_backbone=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None,
                           num_classes=19, aux_loss=False)
    model = maskrcnn_resnet50_fpn(
        weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1 if pretrained else None,
        weights_backbone=None, min_size=CONFIG['height'], max_size=CONFIG['width'],
        box_score_thresh=CONFIG['instance_score_threshold'],
        box_detections_per_img=CONFIG['detections_per_image'])
    box_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(box_features, 9)
    mask_features = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(mask_features, 256, 9)
    return model

def train_epoch(model, loader, optimizer, task):
    model.train()
    if task == 'semantic':
        for module in model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm): module.eval()
    total, count = 0.0, 0
    for images, targets in loader:
        optimizer.zero_grad(set_to_none=True)
        if task == 'semantic':
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            if not (targets != 255).any(): continue
            loss = F.cross_entropy(model(images)['out'], targets, ignore_index=255)
        else:
            images = [x.to(DEVICE) for x in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss = sum(model(images, targets).values())
        if not torch.isfinite(loss): raise RuntimeError(f'Training loss is not finite: {loss.item()}')
        loss.backward()
        optimizer.step()
        total += loss.item() * len(images)
        count += len(images)
    if count == 0: raise RuntimeError('No valid training samples.')
    return total / count

@torch.inference_mode()
def evaluate_semantic(model, records, destination):
    model.eval()
    matrix = torch.zeros((19, 19), dtype=torch.int64)
    for row in records:
        image = Image.open(row['image']).convert('RGB')
        x = TF.to_tensor(image.resize((CONFIG['width'], CONFIG['height']), Image.Resampling.BILINEAR))
        x = TF.normalize(x, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        logits = model(x.unsqueeze(0).to(DEVICE))['out']
        prediction = F.interpolate(logits, size=(image.height, image.width), mode='bilinear', align_corners=False).argmax(1)[0].cpu()
        target = torch.from_numpy(LUT[np.array(Image.open(row['semantic']))].astype(np.int64))
        valid = target != 255
        matrix += torch.bincount(19 * target[valid] + prediction[valid], minlength=361).reshape(19, 19)
    intersection = matrix.diag().double()
    union = matrix.sum(0) + matrix.sum(1) - intersection
    iou = torch.where(union > 0, intersection / union, torch.nan)
    scores = {'miou': iou.nanmean().item(),
              'pixel_accuracy': (intersection.sum() / matrix.sum().clamp_min(1)).item(),
              'per_class_iou': {name: float(v) if torch.isfinite(v) else None for name, v in zip(CLASS_NAMES, iou)}}
    destination.mkdir(parents=True, exist_ok=True)
    np.save(destination / 'confusion.npy', matrix.numpy())
    (destination / 'semantic.json').write_text(json.dumps(scores, indent=2))
    return scores

@torch.inference_mode()
def evaluate_instance(model, records, destination):
    from cityscapesscripts.evaluation import evalInstanceLevelSemanticLabeling as official
    model.eval()
    destination = destination.resolve()
    destination.mkdir(parents=True, exist_ok=True)
    # Stage evaluator masks locally; keep only scores and exports in destination.
    staging = SCRATCH_ROOT / f'{destination.parent.name}_{destination.name}'
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True, exist_ok=True)
    staging = staging.resolve()
    predictions, ground_truth = [], []
    for row in records:
        image = Image.open(row['image']).convert('RGB')
        x = TF.to_tensor(image.resize((CONFIG['width'], CONFIG['height']), Image.Resampling.BILINEAR))
        result = model([x.to(DEVICE)])[0]
        lines = []
        for index, (mask, label, score) in enumerate(zip(result['masks'], result['labels'], result['scores'])):
            if score.item() < CONFIG['instance_score_threshold']: continue
            # Restore one mask at a time to avoid allocating all masks at full resolution on GPU.
            mask = F.interpolate(mask.unsqueeze(0), size=(image.height, image.width), mode='bilinear', align_corners=False)[0, 0]
            binary = (mask > CONFIG['mask_threshold']).cpu().numpy().astype(np.uint8)
            name = f"{row['key']}_{index:03d}.png"
            Image.fromarray(binary * 255).save(staging / name)
            lines.append(f'{name} {MODEL_TO_THING[int(label)]} {float(score):.8f}')
        txt = staging / f"{row['key']}.txt"
        txt.write_text('\n'.join(lines) + ('\n' if lines else ''))
        predictions.append(str(txt.resolve()))
        ground_truth.append(str(Path(row['instance']).resolve()))
    args = copy.deepcopy(official.args)
    args.predictionPath = str(staging)
    # Key the evaluator's ground truth cache by split fingerprint.
    args.gtInstancesFile = str(SCRATCH_ROOT / f'gtInstances_{SPLIT_HASH[:16]}.json')
    args.exportFile = str(destination / 'official-instance.json')
    args.quiet = True
    official.setInstanceLabels(args)
    # The official script writes matches.json to its working directory.
    previous_directory = Path.cwd()
    try:
        os.chdir(staging)
        results = official.evaluateImgLists(predictions, ground_truth, args)
    finally:
        os.chdir(previous_directory)
    # Remove temporary masks and matches.json after scoring.
    shutil.rmtree(staging, ignore_errors=True)
    return {'ap': float(results['averages']['allAp']), 'ap50': float(results['averages']['allAp50%'])}

def official_instance_scores(export):
    """Read the headline AP values from one official evaluator export."""
    result = json.loads(Path(export).read_text())
    return {'ap': float(result['averages']['allAp']),
            'ap50': float(result['averages']['allAp50%'])}

@torch.inference_mode()
def rescore_instance(run_directory, cap):
    """Evaluate one checkpoint at a new detection cap, reusing a finished export."""
    destination = Path(run_directory) / f'best_validation_cap{cap}'
    export = destination / 'official-instance.json'
    if export.exists():
        print(f'cap={cap}: reusing {export}')
        return official_instance_scores(export)
    checkpoint = torch.load(Path(run_directory) / 'best.pt', map_location='cpu', weights_only=False)
    model = build_model(checkpoint['metadata']['spec'])
    model.load_state_dict(checkpoint['model'])
    model.roi_heads.detections_per_img = cap
    model.to(DEVICE).eval()
    scores = evaluate_instance(model, val_records, destination)
    del model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return scores


In [ ]:
def run_experiment(experiment_id, resume=None):
    spec = EXPERIMENTS[experiment_id]
    if not CONFIG['smoke'] and PROVENANCE['source_commit'].startswith('TBD'):
        raise RuntimeError('Set PROVENANCE["source_commit"] before a reportable run.')
    seed_all(CONFIG['seed'])
    # Add a counter when two runs start in the same minute.
    if resume:
        directory = Path(resume).parent
    else:
        stamp = datetime.now().strftime('%m-%d_%H%M')
        directory = OUTPUT_ROOT / f'{experiment_id}_{stamp}'
        suffix = 2
        while directory.exists():
            directory = OUTPUT_ROOT / f'{experiment_id}_{stamp}_{suffix}'
            suffix += 1
    directory.mkdir(parents=True, exist_ok=True)
    metadata = {'experiment': experiment_id, 'spec': spec, 'config': CONFIG,
                'provenance': PROVENANCE, 'environment': ENVIRONMENT,
                'split_hash': SPLIT_HASH,
                'evaluation_split': 'validation', 'smoke': CONFIG['smoke'],
                'profile': CONFIG['profile']}
    model = build_model(spec).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
    loader = make_loader(spec['task'], spec['augment'])
    start, best, history = 0, -float('inf'), []
    if resume:
        # Load only trusted checkpoints produced by this notebook.
        saved = torch.load(resume, map_location='cpu', weights_only=False)
        assert saved['metadata']['config'] == CONFIG, 'Resume requires the original CONFIG.'
        assert saved['metadata']['spec'] == spec, 'Checkpoint belongs to another experiment.'
        assert saved['metadata']['split_hash'] == SPLIT_HASH, 'Resume requires the original split.'
        model.load_state_dict(saved['model']); optimizer.load_state_dict(saved['optimizer'])
        start, best, history = saved['epoch'] + 1, saved['best'], saved['history']
        random.setstate(saved['rng']['python']); np.random.set_state(saved['rng']['numpy'])
        torch.set_rng_state(saved['rng']['torch'])
        if DEVICE.type == 'cuda' and saved['rng']['cuda'] is not None:
            torch.cuda.set_rng_state_all(saved['rng']['cuda'])
        loader.generator.set_state(saved['rng']['loader'])
    else:
        (directory / 'metadata.json').write_text(json.dumps(metadata, indent=2))
        (directory / 'splits.json').write_text(json.dumps(SPLIT_MANIFEST, indent=2))
        (directory / 'pip-freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
    # A smoke run only checks that the pipeline executes, so 1 epoch is enough.
    if CONFIG['smoke']:
        total_epochs = CONFIG['smoke_epochs']
    elif CONFIG['profile']:
        total_epochs = 1
    else:
        total_epochs = CONFIG['epochs']
    for epoch in range(start, total_epochs):
        begin = time.perf_counter()
        loss = train_epoch(model, loader, optimizer, spec['task'])
        # Time training and validation separately.
        trained = time.perf_counter()
        evaluate = evaluate_semantic if spec['task'] == 'semantic' else evaluate_instance
        # Reuse last_validation; history.csv retains all epoch metrics.
        epoch_dir = directory / 'last_validation'
        scores = evaluate(model, val_records, epoch_dir)
        selection = scores['miou' if spec['task'] == 'semantic' else 'ap']
        if not np.isfinite(selection):
            raise RuntimeError('Selection metric is undefined; expand the validation subset to include valid targets.')
        improved = selection > best
        best = max(best, selection)
        now = time.perf_counter()
        history.append({'epoch': epoch, 'loss': loss,
                        'train_seconds': trained - begin, 'val_seconds': now - trained,
                        'seconds': now - begin,
                        **{k: v for k, v in scores.items() if not isinstance(v, dict)}})
        payload = {'epoch': epoch, 'best': best, 'history': history, 'metadata': metadata,
                   'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                   'rng': {'python': random.getstate(), 'numpy': np.random.get_state(),
                           'torch': torch.get_rng_state(),
                           'cuda': torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else None,
                           'loader': loader.generator.get_state()}}
        # Save on persistent storage; replace last.pt only after a successful write.
        torch.save(payload, directory / 'last.tmp.pt')
        (directory / 'last.tmp.pt').replace(directory / 'last.pt')
        if improved: torch.save(payload, directory / 'best.pt')
        pd.DataFrame(history).to_csv(directory / 'history.csv', index=False)
        # Full precision stays in history.csv; this line is for watching progress.
        shown = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in history[-1].items()}
        shown['train_seconds'] = round(shown['train_seconds'])
        shown['val_seconds'] = round(shown['val_seconds'])
        shown['seconds'] = round(shown['seconds'])
        print(experiment_id, shown)
    best_state = torch.load(directory / 'best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(best_state['model'])
    evaluate = evaluate_semantic if spec['task'] == 'semantic' else evaluate_instance
    final_scores = evaluate(model, val_records, directory / 'best_validation')
    (directory / 'summary.json').write_text(json.dumps({**metadata, 'best_epoch': best_state['epoch'], 'scores': final_scores}, indent=2))
    del model, optimizer
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return directory


## 5. Experiments and results
The reported runs use `smoke=False`, `profile=False`, 5 epochs and all four configurations.

| Mode | Setting | Epochs | Runs | Purpose |
|---|---|---|---|---|
| smoke | `smoke=True` | `smoke_epochs` | S0, S1, S2, I0 | Check the pipeline on 8 training and 4 validation images |
| profile | `smoke=False`, `profile=True` | 1 | S1, I0 | Measure time on the full split |
| full | both `False` | 5 | S0, S1, S2, I0 | Produce report results |

The smoke and profile scores are excluded from the report. Set `PROVENANCE['source_commit']` to a real commit before a full run. The four experiments use one split manifest and verify its fingerprint.

After a Colab reconnect, the cell skips completed runs with `summary.json` and resumes compatible runs from `last.pt`. Each run directory under `OUTPUT_ROOT` stores the split manifest, environment, epoch history, checkpoints and evaluator exports. Figures go to `OUTPUT_ROOT/figures`. Temporary evaluator masks go to `SCRATCH_ROOT` on the runtime disk.


In [ ]:
# Profile one semantic run and the instance run.
PROFILING = CONFIG['profile'] and not CONFIG['smoke']
SELECTED_EXPERIMENTS = ['S1', 'I0'] if PROFILING else ['S0', 'S1', 'S2', 'I0']
def find_existing_run(experiment_id):
    """Return the latest compatible run and whether it is complete."""
    candidates = sorted(OUTPUT_ROOT.glob(f'{experiment_id}_*'), reverse=True)
    for directory in candidates:
        meta_file = directory / 'metadata.json'
        if not meta_file.exists():
            continue
        meta = json.loads(meta_file.read_text())
        # Only reuse a run that would compare against the others on equal terms.
        if meta.get('config') != CONFIG or meta.get('split_hash') != SPLIT_HASH:
            continue
        if (directory / 'summary.json').exists():
            return directory, True
        if (directory / 'last.pt').exists():
            return directory, False
    return None, False

RUN_DIRECTORIES = {}
if RUN_TRAINING:
    for experiment_id in SELECTED_EXPERIMENTS:
        existing, finished = find_existing_run(experiment_id)
        if finished:
            print(f'{experiment_id}: already complete at {existing.name}, skipping.')
            RUN_DIRECTORIES[experiment_id] = str(existing)
            continue
        if existing is not None:
            print(f'{experiment_id}: resuming {existing.name}.')
            RUN_DIRECTORIES[experiment_id] = str(run_experiment(experiment_id, resume=existing / 'last.pt'))
            continue
        print(f'{experiment_id}: starting a new run.')
        RUN_DIRECTORIES[experiment_id] = str(run_experiment(experiment_id))
else:
    print('Training skipped. Set RUN_TRAINING=True to launch', SELECTED_EXPERIMENTS, '.')


In [ ]:
# Recover compatible finished runs after a fresh Colab session.
# Use COMPLETED_RUNS only to include a run rejected by the automatic checks.
COMPLETED_RUNS = [
    # '/content/drive/MyDrive/urban-scene-runs/S0_10-02_0136',
]
discovered = []
for experiment_id in ['S0', 'S1', 'S2', 'I0']:
    directory, finished = find_existing_run(experiment_id)
    if finished:
        discovered.append(str(directory))
REPORT_RUNS = list(dict.fromkeys(list(RUN_DIRECTORIES.values()) + discovered + COMPLETED_RUNS))
missing = [p for p in REPORT_RUNS if not (Path(p) / 'summary.json').exists()]
assert not missing, f'No summary.json in: {missing}'
if not REPORT_RUNS:
    raise RuntimeError(
        f'No finished runs found under {OUTPUT_ROOT}. Check that Drive is mounted and that '
        'CONFIG matches the runs you want to report, or list their paths in COMPLETED_RUNS.')
print('Reporting on:', ', '.join(Path(p).name for p in REPORT_RUNS))
rows = []
history_by_experiment = {}
for path in REPORT_RUNS:
    result = json.loads((Path(path) / 'summary.json').read_text())
    scores = {k: v for k, v in result['scores'].items() if not isinstance(v, dict)}
    if result['spec']['task'] == 'instance':
        final_export = (Path(path) / f'best_validation_cap{REPORT_DETECTION_CAP}' /
                        'official-instance.json')
        if not final_export.exists():
            raise FileNotFoundError(f'Missing final instance export: {final_export}')
        scores = official_instance_scores(final_export)
    rows.append({'experiment': result['experiment'], 'smoke': result['smoke'],
                 'profile': result.get('profile', False),
                 'split_hash': result['split_hash'], 'best_epoch': result['best_epoch'],
                 **scores})
    history = pd.read_csv(Path(path) / 'history.csv')
    history_by_experiment[result['experiment']] = history.copy()
    metric = 'miou' if result['spec']['task'] == 'semantic' else 'ap'
    # Mark points and use integer epoch ticks.
    axis = history.plot(x='epoch', y=metric, marker='o', legend=False,
                        title=f"{result['experiment']}: validation {metric}")
    axis.set_xlabel('epoch'); axis.set_ylabel(metric)
    axis.set_xticks(history['epoch'].astype(int))
    plt.tight_layout(); plt.show()
if rows:
    table = pd.DataFrame(rows)
    # Compare the full fingerprints, then shorten them for display.
    assert table['split_hash'].nunique() == 1, 'Comparison requires identical data splits.'
    table['split_hash'] = table['split_hash'].str[:12]
    if table['smoke'].any():
        print('Smoke rows are pipeline checks and are excluded from report results.')
    if table['profile'].any():
        print('Profile rows are timing runs and are excluded from report results.')
    display(table)
    table.to_csv(OUTPUT_ROOT / 'comparison.csv', index=False)

    # Plot validation scores across epochs.
    figure, axes = plt.subplots(2, 1, figsize=(3.45, 2.95), sharex=True)
    semantic_styles = {
        'S0': {'color': '#1f77b4', 'marker': 'o', 'linestyle': '-'},
        'S1': {'color': '#d62728', 'marker': 's', 'linestyle': '--'},
        'S2': {'color': '#2ca02c', 'marker': '^', 'linestyle': '-.'},
    }
    for experiment in ('S0', 'S1', 'S2'):
        history = history_by_experiment[experiment]
        axes[0].plot(history['epoch'] + 1, history['miou'], label=experiment,
                     linewidth=1.2, markersize=3.8, **semantic_styles[experiment])
    axes[0].set_title('Semantic validation mIoU', fontsize=9.5, pad=3)
    axes[0].set_ylabel('mIoU', fontsize=9)
    axes[0].set_ylim(0.33, 0.505)
    axes[0].legend(ncol=3, loc='lower right', frameon=False, fontsize=8,
                   handlelength=1.8, columnspacing=0.8)
    instance_history = history_by_experiment['I0']
    axes[1].plot(instance_history['epoch'] + 1, instance_history['ap'],
                 color='#9467bd', marker='D', linewidth=1.2, markersize=3.8)
    axes[1].set_title('Instance validation AP at limit 50', fontsize=9.5, pad=3)
    axes[1].set_xlabel('Epoch', fontsize=9)
    axes[1].set_ylabel('AP', fontsize=9)
    axes[1].set_ylim(0.078, 0.126)
    axes[1].set_xticks(range(1, 6))
    for axis in axes:
        style_report_axis(axis, grid=True)
        axis.tick_params(axis='both', labelsize=8.5)
    figure.tight_layout(pad=0.55, h_pad=0.45)
    figure.savefig(FIGURE_DIR / 'training-curves.png', dpi=300,
                   bbox_inches='tight', pad_inches=0.035)
    plt.show()
else:
    print('No completed runs selected. No numerical results are available.')

# Estimate budgets from separate training and validation timings.
for path in REPORT_RUNS:
    history = pd.read_csv(Path(path) / 'history.csv')
    if 'train_seconds' not in history.columns:
        continue
    result = json.loads((Path(path) / 'summary.json').read_text())
    first = history.iloc[0]
    label = result['experiment']
    print(f"\n{label}: train {first['train_seconds']:.0f}s + validation "
          f"{first['val_seconds']:.0f}s = {first['seconds']:.0f}s per epoch")
    for budget in (5, 10):
        # Each run also evaluates the best checkpoint again once at the end.
        total = (first['seconds'] * budget + first['val_seconds']) / 3600
        print(f"  {budget:2d} epochs: {total:.1f} h for this experiment")

# The later analysis cells read the instance run's official export and best checkpoint.
INSTANCE_RUN = next((p for p in REPORT_RUNS
                     if json.loads((Path(p) / 'summary.json').read_text())['spec']['task'] == 'instance'), None)
if INSTANCE_RUN is None:
    print('No instance run among the reported runs; the crowding cells below will be skipped.')


### Inspect predictions
Use the same validation image for each semantic model. Figures hide instance predictions below 0.5 for readability. Official AP scoring uses a 0.05 export threshold; the display setting does not change the score.


In [ ]:
@torch.inference_mode()
def show_prediction(run_directory, index=0, save_as=None):
    checkpoint = torch.load(Path(run_directory) / 'best.pt', map_location='cpu', weights_only=False)
    # Match the architecture fields. Epoch count and mode may differ for figures.
    saved = checkpoint['metadata']['config']
    for key in ('height', 'width', 'pretrained', 'detections_per_image', 'instance_score_threshold'):
        assert saved[key] == CONFIG[key], f'{key} differs from the checkpoint; restore it first.'
    spec = checkpoint['metadata']['spec']
    model = build_model(spec).to(DEVICE)
    model.load_state_dict(checkpoint['model']); model.eval()
    row = val_records[index]
    original = Image.open(row['image']).convert('RGB')
    image = original.resize((CONFIG['width'], CONFIG['height']), Image.Resampling.BILINEAR)
    x = TF.to_tensor(image)
    fig, axes = plt.subplots(1, 3, figsize=REPORT_WIDE_SIZE)
    axes[0].imshow(original); axes[0].set_title('Input')
    if spec['task'] == 'semantic':
        x = TF.normalize(x, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        logits = model(x[None].to(DEVICE))['out']
        pred = F.interpolate(logits, size=(original.height, original.width), mode='bilinear', align_corners=False).argmax(1)[0].cpu().numpy()
        axes[1].imshow(semantic_rgb(LUT[np.array(Image.open(row['semantic']))]))
        axes[2].imshow(semantic_rgb(pred))
    else:
        raw = np.array(Image.open(row['instance']))
        gt = np.zeros((*raw.shape, 3), dtype=np.uint8)
        rng = np.random.default_rng(CONFIG['seed'])
        for ident in np.unique(raw):
            if ident >= 1000 and int(ident) // 1000 in THING_TO_MODEL:
                gt[raw == ident] = rng.integers(30, 256, 3)
        axes[1].imshow(gt)
        pred = model([x.to(DEVICE)])[0]
        overlay = np.asarray(image).copy()
        for mask, score in zip(pred['masks'].cpu().numpy(), pred['scores'].cpu().numpy()):
            if score >= 0.5:
                region = mask[0] > CONFIG['mask_threshold']
                overlay[region] = (0.5 * overlay[region] + 0.5 * rng.integers(30, 256, 3)).astype(np.uint8)
        axes[2].imshow(overlay)
    axes[1].set_title('Ground truth'); axes[2].set_title('Best checkpoint prediction')
    for ax in axes: ax.axis('off')
    plt.tight_layout(pad=0.35, w_pad=0.35)
    if save_as is not None:
        fig.savefig(save_as, bbox_inches='tight', pad_inches=0.035)
    plt.show()
    del model

# Same validation image across the semantic runs, which is what the report compares.
FIGURE_DIR = OUTPUT_ROOT / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
for path in REPORT_RUNS:
    result = json.loads((Path(path) / 'summary.json').read_text())
    if result['spec']['task'] != 'semantic':
        continue
    print(result['experiment'])
    show_prediction(path, index=0, save_as=FIGURE_DIR / f"semantic-{result['experiment']}.png")


### Select the detection cap and compare errors by class
The AP/AP50 ratio ranges from 0.58 to 0.65 for bus, train, truck and car. It ranges from 0.25 to 0.35 for person, bicycle, motorcycle and rider. A lower ratio can reflect errors at stricter IoU thresholds. Recall, score ranking, duplicate detections and class errors also affect it.

The next cell checks whether contact between instances of the same class follows this pattern. It uses all 500 validation images. The analysis uses the highest tested cap, verifies that it does not truncate predictions, then uses that export for the class, contact and geometry results.


In [ ]:
from scipy import ndimage

# Define the output directory so this cell can run independently.
FIGURE_DIR = OUTPUT_ROOT / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def contact_counts(raw, radius=3):
    """Per class (touching, total) instance counts for one ground truth instance map.

    An instance counts as touching when another instance of the same class is separated
    from it by fewer than `radius` empty pixels: dilating a mask `radius` times bridges a
    gap of `radius - 1`. Work happens inside each instance's bounding box, so the cost
    scales with instance area rather than image area.
    """
    counts = {}
    for label in THING_LABELS:
        class_mask = (raw >= 1000) & (raw // 1000 == label.id)
        identifiers = np.unique(raw[class_mask])
        touching = 0
        for identifier in identifiers:
            own = raw == identifier
            ys, xs = np.where(own)
            y0, y1 = max(ys.min() - radius, 0), min(ys.max() + radius + 1, raw.shape[0])
            x0, x1 = max(xs.min() - radius, 0), min(xs.max() + radius + 1, raw.shape[1])
            crop = own[y0:y1, x0:x1]
            others = class_mask[y0:y1, x0:x1] & ~crop
            if others.any() and (ndimage.binary_dilation(crop, iterations=radius) & others).any():
                touching += 1
        counts[label.name] = (touching, len(identifiers))
    return counts

def contact_rates(records, radius=3):
    """Aggregate class contact counts and totals for each image."""
    totals = {x.name: [0, 0] for x in THING_LABELS}
    per_image = []
    for row in records:
        counts = contact_counts(np.array(Image.open(row['instance'])), radius)
        for name, (touching, total) in counts.items():
            totals[name][0] += touching
            totals[name][1] += total
        per_image.append({'key': row['key'],
                          **{name: touching for name, (touching, _) in counts.items()}})
    table = pd.DataFrame([{'class': name, 'instances': total,
                           'contact_rate': touching / total if total else np.nan}
                          for name, (touching, total) in totals.items()])
    return table, pd.DataFrame(per_image)

def per_class_ap(export):
    """AP for each class from one selected official evaluator export."""
    export = Path(export)
    classes = json.loads(export.read_text())['averages']['classes']
    return pd.DataFrame([{'class': name, 'ap': v['ap'], 'ap50': v['ap50%']}
                         for name, v in classes.items()])

# Use the full validation split unless a smaller diagnostic sample is requested.
CROWDING_SAMPLE = None
sample = (val_records if CROWDING_SAMPLE is None or len(val_records) <= CROWDING_SAMPLE
          else random.Random(CONFIG['seed']).sample(val_records, CROWDING_SAMPLE))
if len(sample) < len(val_records):
    print(f'Contact rates use {len(sample)} of {len(val_records)} validation images, '
          f'while AP covers all of them; report the mismatch alongside the correlation.')
if INSTANCE_RUN is None:
    raise RuntimeError('This cell needs a finished instance run; none was found above.')
# Select one cached cap export for all downstream analyses.
CONFIGURED_CAP = CONFIG['detections_per_image']
CAPS_TO_COMPARE = sorted({100, REPORT_DETECTION_CAP})
cap_exports = {CONFIGURED_CAP: Path(INSTANCE_RUN) / 'best_validation' / 'official-instance.json'}
cap_scores = {CONFIGURED_CAP: official_instance_scores(cap_exports[CONFIGURED_CAP])}
for cap in CAPS_TO_COMPARE:
    print(f'Rescoring at cap={cap}; a cached official export is reused when present.')
    cap_scores[cap] = rescore_instance(INSTANCE_RUN, cap)
    cap_exports[cap] = Path(INSTANCE_RUN) / f'best_validation_cap{cap}' / 'official-instance.json'
cap_table = pd.DataFrame([{'detections_per_image': cap, **scores}
                          for cap, scores in sorted(cap_scores.items())])
baseline_ap = cap_scores[CONFIGURED_CAP]['ap']
cap_table['ap_gain'] = cap_table['ap'] - baseline_ap
selected_cap = REPORT_DETECTION_CAP
INSTANCE_EVAL_EXPORT = cap_exports[selected_cap]
display(cap_table)
print(f'All downstream instance analysis uses cap={selected_cap}: {INSTANCE_EVAL_EXPORT}')
cap_table.to_csv(FIGURE_DIR / 'detection-cap.csv', index=False)
(FIGURE_DIR / 'detection-cap.json').write_text(json.dumps(
    {'run': str(INSTANCE_RUN), 'split_hash': SPLIT_HASH, 'selected_cap': selected_cap,
     'scores_by_cap': {str(k): v for k, v in cap_scores.items()}}, indent=2))
contact_table, per_image_contacts = contact_rates(sample)
instance_classes = per_class_ap(INSTANCE_EVAL_EXPORT)
# Use the selected instance export for correlations across tasks.
cross_task = instance_classes[['class', 'ap']].copy()
cross_correlations = {}
for path in REPORT_RUNS:
    summary = json.loads((Path(path) / 'summary.json').read_text())
    if summary['spec']['task'] != 'semantic':
        continue
    experiment = summary['experiment']
    semantic_export = json.loads((Path(path) / 'best_validation' / 'semantic.json').read_text())
    values = semantic_export['per_class_iou']
    cross_task[experiment] = cross_task['class'].map(values)
    cross_correlations[experiment] = float(cross_task['ap'].corr(
        cross_task[experiment], method='spearman'))
display(cross_task)
print('Spearman, instance AP against semantic IoU:',
      ', '.join(f'{key} {value:.3f}' for key, value in sorted(cross_correlations.items())))
cross_task.to_csv(FIGURE_DIR / 'cross-task.csv', index=False)
(FIGURE_DIR / 'cross-task.json').write_text(json.dumps(
    {'instance_export': str(INSTANCE_EVAL_EXPORT), 'detection_cap': selected_cap,
     'spearman_by_semantic_run': cross_correlations,
     'classes': cross_task.to_dict('records')}, indent=2))
crowding = instance_classes.merge(contact_table, on='class')
crowding['ap_ratio'] = crowding['ap'] / crowding['ap50']
crowding = crowding.sort_values('ap_ratio', ascending=False)
spearman = crowding['contact_rate'].corr(crowding['ap_ratio'], method='spearman')
# Use an exact permutation test for the eight classes.
from itertools import permutations
from math import factorial
observed = abs(spearman)
ratios = pd.Series(crowding['ap_ratio'].to_numpy())
contacts = crowding['contact_rate'].to_numpy()
orders = list(permutations(range(len(crowding))))
extreme = sum(abs(pd.Series(contacts[list(o)]).corr(ratios, method='spearman')) >= observed - 1e-12
              for o in orders)
p_value = extreme / factorial(len(crowding))
display(crowding)
print(f'Spearman, contact rate against AP/AP50 ratio: {spearman:.3f} '
      f'(exact permutation p = {p_value:.3f}, n = {len(crowding)})')
crowding.to_csv(FIGURE_DIR / 'crowding.csv', index=False)
(FIGURE_DIR / 'crowding.json').write_text(json.dumps(
    {'run': str(INSTANCE_RUN), 'evaluation_export': str(INSTANCE_EVAL_EXPORT),
     'detection_cap': selected_cap, 'split_hash': SPLIT_HASH, 'sample_images': len(sample),
     'contact_radius': 3, 'spearman_contact_vs_ap_ratio': float(spearman),
     'permutation_p_value': float(p_value),
     'classes': crowding.to_dict('records')}, indent=2))

figure, axis = plt.subplots(figsize=REPORT_SINGLE_SIZE)
axis.scatter(crowding['contact_rate'], crowding['ap_ratio'],
             s=20, color='#1f77b4', zorder=3)
contact_offsets = {
    'truck': (3, -10), 'motorcycle': (-42, 0),
}
for _, r in crowding.iterrows():
    axis.annotate(r['class'], (r['contact_rate'], r['ap_ratio']), fontsize=7.0,
                  xytext=contact_offsets.get(r['class'], (3, 3)),
                  textcoords='offset points')
axis.set_xlabel('Ground truth contact rate'); axis.set_ylabel('AP / AP50')
axis.set_title('AP/AP50 vs same class contact', pad=5)
style_report_axis(axis, grid=True)
figure.tight_layout(pad=0.45)
figure.savefig(FIGURE_DIR / 'crowding-scatter.png', bbox_inches='tight', pad_inches=0.035)
plt.show()
print('Wrote', FIGURE_DIR / 'crowding.csv', '/ crowding.json / crowding-scatter.png')

This scene contains many touching instances. The model misses many distant objects.


In [ ]:
@torch.inference_mode()
def show_instance_separation(run_directory, row, score_threshold=0.5, classes=('person', 'bicycle'),
                             save_as=None):
    """Plot ground truth and prediction with instance counts for selected classes."""
    checkpoint = torch.load(Path(run_directory) / 'best.pt', map_location='cpu', weights_only=False)
    model = build_model(checkpoint['metadata']['spec']).to(DEVICE)
    model.load_state_dict(checkpoint['model']); model.eval()
    wanted = {x.id for x in THING_LABELS if x.name in classes}
    rng = np.random.default_rng(CONFIG['seed'])
    original = Image.open(row['image']).convert('RGB')
    image = original.resize((CONFIG['width'], CONFIG['height']), Image.Resampling.BILINEAR)
    raw = np.array(Image.open(row['instance']).resize(image.size, Image.Resampling.NEAREST))

    truth = np.asarray(image).copy()
    expected = 0
    for identifier in np.unique(raw):
        if identifier >= 1000 and int(identifier) // 1000 in wanted:
            region = raw == identifier
            truth[region] = (0.4 * truth[region] + 0.6 * rng.integers(30, 256, 3)).astype(np.uint8)
            expected += 1

    prediction = model([TF.to_tensor(image).to(DEVICE)])[0]
    overlay = np.asarray(image).copy()
    found = 0
    for mask, label, score in zip(prediction['masks'].cpu().numpy(),
                                  prediction['labels'].cpu().numpy(),
                                  prediction['scores'].cpu().numpy()):
        if score < score_threshold or MODEL_TO_THING[int(label)] not in wanted:
            continue
        region = mask[0] > CONFIG['mask_threshold']
        overlay[region] = (0.4 * overlay[region] + 0.6 * rng.integers(30, 256, 3)).astype(np.uint8)
        found += 1

    fig, axes = plt.subplots(1, 3, figsize=REPORT_WIDE_SIZE)
    for ax, picture, title in zip(axes, [image, truth, overlay],
                                  ['Input', f'Ground truth: {expected}',
                                   f'Prediction: {found} at score >= {score_threshold}']):
        ax.imshow(picture); ax.set_title(title); ax.axis('off')
    plt.tight_layout(pad=0.35, w_pad=0.35)
    if save_as is not None:
        fig.savefig(save_as, bbox_inches='tight', pad_inches=0.035)
    plt.show()
    del model
    return expected, found

if INSTANCE_RUN is None:
    raise RuntimeError('This cell needs a finished instance run; none was found above.')
CROWDED_CLASSES = ['person', 'bicycle']
ranked = per_image_contacts.assign(total=per_image_contacts[CROWDED_CLASSES].sum(axis=1))
ranked = ranked.sort_values('total', ascending=False)
by_key = {row['key']: row for row in val_records}
counts = []
for rank, key in enumerate(ranked['key'].head(2), start=1):
    target = FIGURE_DIR / f'crowding-scene-{rank}.png'
    expected, found = show_instance_separation(INSTANCE_RUN, by_key[key],
                                               classes=tuple(CROWDED_CLASSES), save_as=target)
    counts.append({'rank': rank, 'key': key, 'figure': target.name,
                   'ground_truth': expected, 'predicted_at_0.5': found})
display(pd.DataFrame(counts))
(FIGURE_DIR / 'crowding-scenes.json').write_text(json.dumps(
    {'run': str(INSTANCE_RUN), 'classes': CROWDED_CLASSES,
     'display_threshold': 0.5, 'scenes': counts}, indent=2))
print('Wrote two scene figures and', FIGURE_DIR / 'crowding-scenes.json')

### Measure object size
Measure the median area, width, height and aspect ratio of each class at model input size. Compare each measure with AP/AP50 across the eight thing classes.

Compactness based on perimeter is unstable for masks only a few pixels wide. Area and bounding box dimensions remain measurable at this resolution.


In [ ]:
def instance_shape_stats(records, scale_to_input=True):
    """Measure instance geometry at model input size by default."""
    rows = []
    for row in records:
        raw = np.array(Image.open(row['instance']))
        if scale_to_input:
            raw = np.array(Image.fromarray(raw).resize(
                (CONFIG['width'], CONFIG['height']), Image.Resampling.NEAREST))
        for identifier in np.unique(raw):
            if identifier < 1000:
                continue
            label_id = int(identifier) // 1000
            if label_id not in THING_TO_MODEL:
                continue
            mask = raw == identifier
            area = int(mask.sum())
            if area == 0:
                continue
            ys, xs = np.where(mask)
            width = int(xs.max() - xs.min() + 1)
            height = int(ys.max() - ys.min() + 1)
            rows.append({'class': THING_ID_TO_NAME[label_id], 'area': area,
                         'width': width, 'height': height,
                         'aspect': max(width, height) / min(width, height)})
    return pd.DataFrame(rows)

if INSTANCE_RUN is None:
    raise RuntimeError('This cell needs a finished instance run; none was found above.')
THING_ID_TO_NAME = {x.id: x.name for x in THING_LABELS}
shapes = instance_shape_stats(val_records)
geometry = (shapes.groupby('class')
            .agg(instances=('area', 'size'), median_area=('area', 'median'),
                 median_width=('width', 'median'), median_height=('height', 'median'),
                 median_aspect=('aspect', 'median'))
            .reset_index())
size_shape = crowding.merge(geometry, on='class')
display(size_shape[['class', 'ap', 'ap_ratio', 'contact_rate', 'median_area',
                    'median_width', 'median_height', 'median_aspect']])

print('Spearman against the AP/AP50 ratio, eight thing classes:')
correlations = {}
for column in ['median_area', 'median_width', 'median_height', 'median_aspect',
               'contact_rate']:
    rho = size_shape[column].corr(size_shape['ap_ratio'], method='spearman')
    observed = abs(rho)
    values = pd.Series(size_shape['ap_ratio'].to_numpy())
    candidate = size_shape[column].to_numpy()
    extreme = sum(abs(pd.Series(candidate[list(o)]).corr(values, method='spearman')) >= observed - 1e-12
                  for o in orders)
    p = extreme / factorial(len(size_shape))
    correlations[column] = {'spearman': float(rho), 'permutation_p': float(p)}
    print(f'  {column:16s} rho {rho:+.3f}   p {p:.3f}')
print('\nCoefficients and exact p values use eight classes.')

size_shape.to_csv(FIGURE_DIR / 'size-shape.csv', index=False)
(FIGURE_DIR / 'size-shape.json').write_text(json.dumps(
    {'run': str(INSTANCE_RUN), 'split_hash': SPLIT_HASH, 'images': len(val_records),
     'measured_at': f"{CONFIG['height']}x{CONFIG['width']}",
     'correlations': correlations, 'classes': size_shape.to_dict('records')}, indent=2))

figure, axes = plt.subplots(1, 2, figsize=REPORT_DOUBLE_SIZE)
for axis, column, label in zip(axes, ['median_width', 'median_area'],
                               ['median instance width at input size (px)',
                                'median instance area at input size (px)']):
    axis.scatter(size_shape[column], size_shape['ap_ratio'])
    for _, r in size_shape.iterrows():
        axis.annotate(r['class'], (r[column], r['ap_ratio']), fontsize=7.0,
                      xytext=(3, 3), textcoords='offset points')
    axis.set_xlabel(label); axis.set_ylabel('AP / AP50')
    if column == 'median_area':
        axis.set_xscale('log')
    style_report_axis(axis, grid=True)
figure.suptitle('AP/AP50 ratio against measured instance geometry', fontsize=9.5)
figure.tight_layout(pad=0.45)
figure.savefig(FIGURE_DIR / 'size-shape.png', bbox_inches='tight', pad_inches=0.035)
plt.show()

figure, axis = plt.subplots(figsize=REPORT_SINGLE_SIZE)
axis.scatter(size_shape['median_width'], size_shape['ap_ratio'])
width_offsets = {'rider': (-20, 3), 'motorcycle': (3, -10)}
for _, row in size_shape.iterrows():
    axis.annotate(row['class'], (row['median_width'], row['ap_ratio']), fontsize=7.0,
                  xytext=width_offsets.get(row['class'], (3, 3)),
                  textcoords='offset points')
axis.set_xlabel('Median instance width at 256x512 (px)')
axis.set_ylabel('AP / AP50')
axis.set_title('AP/AP50 vs instance width', pad=5)
style_report_axis(axis, grid=True)
figure.tight_layout(pad=0.45)
figure.savefig(FIGURE_DIR / 'size-width.png', bbox_inches='tight', pad_inches=0.035)
plt.show()

### Audit how often each detection cap binds
Count detections above the export threshold for caps 50, 100 and 200. This uses the selected I0 checkpoint and changes no weights. Run this diagnostic once during the final execution and save its table to Drive.


In [ ]:
@torch.inference_mode()
def detection_counts(run_directory, records, cap=None):
    """Detections above the export threshold per image, with the cap optionally raised."""
    checkpoint = torch.load(Path(run_directory) / 'best.pt', map_location='cpu', weights_only=False)
    model = build_model(checkpoint['metadata']['spec'])
    if cap is not None:
        model.roi_heads.detections_per_img = cap
    model.load_state_dict(checkpoint['model'])
    model.to(DEVICE).eval()
    counts = []
    for row in records:
        image = Image.open(row['image']).convert('RGB').resize(
            (CONFIG['width'], CONFIG['height']), Image.Resampling.BILINEAR)
        result = model([TF.to_tensor(image).to(DEVICE)])[0]
        counts.append(int((result['scores'] >= CONFIG['instance_score_threshold']).sum()))
    del model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return np.array(counts)

RUN_CAP_DIAGNOSTIC = False
CAP_DIAGNOSTIC_FILE = FIGURE_DIR / 'detection-counts.csv'
if RUN_CAP_DIAGNOSTIC:
    if INSTANCE_RUN is None:
        raise RuntimeError('A completed I0 run is required for the cap diagnostic.')
    diagnostic_rows = []
    for cap in (50, 100, 200):
        counts = detection_counts(INSTANCE_RUN, val_records, cap=cap)
        diagnostic_rows.append({
            'cap': cap,
            'mean_detections': counts.mean(),
            'median_detections': np.median(counts),
            'max_detections': counts.max(),
            'fraction_hitting_cap': (counts >= cap).mean(),
        })
    cap_diagnostic = pd.DataFrame(diagnostic_rows)
    cap_diagnostic.to_csv(CAP_DIAGNOSTIC_FILE, index=False)
elif CAP_DIAGNOSTIC_FILE.exists():
    cap_diagnostic = pd.read_csv(CAP_DIAGNOSTIC_FILE)
else:
    raise FileNotFoundError('Run the cap diagnostic once before loading its table.')

display(cap_diagnostic)
selected_row = cap_diagnostic.loc[cap_diagnostic['cap'] == selected_cap].iloc[0]
assert selected_row.fraction_hitting_cap == 0, 'Selected cap still truncates predictions.'
print(f'Selected cap={selected_cap}: {selected_row.fraction_hitting_cap:.1%} of images hit the cap.')

### Published context
DeepLabV3 with a ResNet101 backbone, output stride 8, 90,000 iterations and 769x769 crops reports 77.82 validation mIoU on Cityscapes. Mask R-CNN with ResNet50-FPN reports 31.5 validation AP at full resolution. These settings use larger inputs and longer schedules than our runs, which use 256x512 inputs for 5 epochs. We use S0 with S1 and S1 with S2 for direct comparisons.

Sources: Chen et al., *Rethinking Atrous Convolution for Semantic Image Segmentation*, 2017; He et al., *Mask R-CNN*, ICCV 2017.


In [ ]:
published_context = pd.DataFrame([
    {
        'method': 'DeepLabV3-ResNet101',
        'setting': 'OS8, 90k iterations, 769x769 crops',
        'metric': 'validation mIoU',
        'published_result': 0.7782,
    },
    {
        'method': 'Mask R-CNN-ResNet50-FPN',
        'setting': 'full resolution images',
        'metric': 'validation AP',
        'published_result': 0.3150,
    },
])
display(published_context)

## 6. Discussion

### Semantic results and efficiency
After 5 epochs at 256x512 input, S0 reaches 0.4899 mIoU. S1 scores 0.0377 lower, has lower IoU on 16 of 19 classes and takes 1.9 times as long per epoch. S2 removes horizontal flipping from S1, recovers 0.0330 mIoU and improves 15 classes. All three semantic runs select the fourth epoch (index 3). Under this budget, FCN gives the best balance between accuracy and efficiency. The flip comparison uses one seed.

### Metrics and errors across tasks
S2 has the highest pixel accuracy at 0.9076, while S0 has the highest mIoU. Mean IoU ranges from 0.52 to 0.55 for stuff classes and from 0.36 to 0.41 for thing classes, showing how large regions can dominate pixel accuracy. Across the eight shared thing classes, rank correlations between semantic IoU and instance AP range from 0.55 to 0.71. Car ranks first and train last on both tasks. Bicycle ranks third by S0 IoU and sixth by AP, while bus ranks fourth by S0 IoU and second by AP. Instance scoring also measures detection, separation, localisation and confidence ranking errors.

### Instance error analysis
The limit of 50 detections binds on 73.6% of validation images. At 100 detections, 15.6% of images still reach the limit. A limit of 200 raises AP from 0.12235 to 0.12399, and no image reaches the new limit. Caps 100 and 200 differ by less than 0.00001 AP, so the analysis uses 200 to avoid truncation. Person, bicycle, motorcycle and rider have the lowest AP/AP50 ratios. Contact between objects of the same class has a weak association with this ratio (Spearman rho = -0.286, p = 0.501). Median width has a stronger association (rho = 0.810, p = 0.022), and median area gives rho = 0.738 (p = 0.046). This analysis contains eight classes, so the width result needs direct testing at higher resolution. Train has 23 validation instances, so its AP50 and ratio are less stable than those of larger classes.

### Strengths, limits and next experiment
The official split, instance evaluator and shared split fingerprint keep the comparisons consistent. Every cap uses the same instance checkpoint, and the class statistics cover all 500 validation images. The validation split is used for checkpoint selection and reporting, which can overestimate performance. Each configuration has one seed. Models were trained for 5 epochs at 256x512 without a learning rate schedule or parameter search. Mask R-CNN treats group regions as background, and I0 continues to improve in the final epoch. Next, repeat S0, S2 and I0 at 512x1024 with three seeds and a longer scheduled run. Keep the split, pretrained weights and evaluator fixed. Compare improvements for narrow classes and large vehicles before one final test set evaluation.


## 7. Credits
TorchVision supplies the models and pretrained weights. cityscapesScripts supplies label definitions and instance scoring. We adapted the data pipeline, ran the comparisons and analysed the errors.

Cordts et al., *The Cityscapes Dataset for Semantic Urban Scene Understanding*, CVPR 2016. Long et al., *Fully Convolutional Networks for Semantic Segmentation*, CVPR 2015. Chen et al., *Rethinking Atrous Convolution for Semantic Image Segmentation*, 2017. He et al., *Mask R-CNN*, ICCV 2017. Method and dataset references are listed in README.md.
